# Notebook 2 — Near-duplicate detection across splits (pHash + dHash)**Why this matters.** The corpus was assembled from five public sources with node-duplication step anywhere in the repo (FACT_CHECK D8). Sources overlap: ISICimages appear in several derived datasets, and PAD-UFES-20 contains multiple imagesof the same lesion. If a train image also appears in test, the reported accuracy isoptimistic and the headline number cannot be defended.**Method.** 64-bit perceptual hash (pHash, DCT-based) and difference hash (dHash)for every image in the 5 paper classes across train/val/test. Pairs are flagged whenthe Hamming distance is 0 (exact hash match), <= 5, or <= 8, and reported only whenthe pair *crosses* splits.CPU only. Expect ~5-12 min for ~6.6k images.**Outputs:** `dup_pairs.csv`, `dup_summary.json`, `montage_closest_pairs.png`,and — if cross-split duplicates are found — `test_clean_manifest.csv`.

In [ ]:
!pip install -q imagehash

In [ ]:
# ============================ EDIT THESE ============================DATA_ROOT = '/kaggle/input/datasets/jainmanav20/ipd-skin-dataset/IPD_Datasets_Final_Split'OUT_DIR   = '/kaggle/working'# ====================================================================import json, itertools, timefrom pathlib import Pathfrom collections import Counter, defaultdictimport numpy as npimport pandas as pdfrom PIL import Imageimport imagehashimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltSEED = 42np.random.seed(SEED)DATA_ROOT = Path(DATA_ROOT); OUT = Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)# Checkpoint class order (same 5 classes as Notebook 1).CLASSES = ['Basal_Cell_Carcinoma', 'Eczema', 'Keratosis', 'Melanoma', 'Tinea']SPLITS  = ['train', 'val', 'test']EXTS    = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}records = []for split in SPLITS:    for cls in CLASSES:        d = DATA_ROOT/split/cls        if not d.is_dir():            print(f'WARNING: missing {d}')            continue        for f in sorted(d.iterdir()):            if f.suffix.lower() in EXTS:                records.append(dict(path=str(f), split=split, cls=cls, name=f.name))df = pd.DataFrame(records)print(f'indexed {len(df)} images across {len(SPLITS)} splits x {len(CLASSES)} classes')print(df.groupby(['split','cls']).size().unstack(fill_value=0))print('\nper split:', df.groupby('split').size().to_dict())

In [ ]:
HASH_SIZE = 8   # 64-bit hashest0 = time.time()ph, dh, failed = [], [], []for i, p in enumerate(df['path'].tolist()):    try:        with Image.open(p) as im:            im = im.convert('RGB')            ph.append(str(imagehash.phash(im, hash_size=HASH_SIZE)))            dh.append(str(imagehash.dhash(im, hash_size=HASH_SIZE)))    except Exception as e:        ph.append(None); dh.append(None); failed.append((p, repr(e)))    if (i+1) % 1000 == 0:        print(f'  {i+1}/{len(df)}  ({time.time()-t0:.0f}s)')df['phash'] = phdf['dhash'] = dhprint(f'\nhashed {len(df)-len(failed)} images in {time.time()-t0:.0f}s; failures: {len(failed)}')for p, e in failed[:10]:    print('  FAILED', p, e)df_ok = df[df['phash'].notna()].reset_index(drop=True)df_ok.to_csv(OUT/'image_hashes.csv', index=False)print(f'wrote image_hashes.csv ({len(df_ok)} rows)')

## Pairwise Hamming distancesHashes are unpacked to a bit matrix so all pairs can be compared with matrixarithmetic instead of a Python double loop (~6.6k x 6.6k is trivial this way).Only **cross-split** pairs are retained.

In [ ]:
def hex_to_bits(hexstr, nbits=64):    v = int(hexstr, 16)    return np.array([(v >> (nbits-1-i)) & 1 for i in range(nbits)], dtype=np.uint8)P = np.stack([hex_to_bits(h) for h in df_ok['phash']]).astype(np.int16)D = np.stack([hex_to_bits(h) for h in df_ok['dhash']]).astype(np.int16)split_arr = df_ok['split'].to_numpy()cls_arr   = df_ok['cls'].to_numpy()path_arr  = df_ok['path'].to_numpy()N = len(df_ok)print(f'bit matrices: pHash {P.shape}, dHash {D.shape}')# Hamming distance via matrix product: d = popcount(a) + popcount(b) - 2*a.bdef hamming_matrix(B):    pc = B.sum(1)    return pc[:, None] + pc[None, :] - 2 * (B @ B.T)t0 = time.time()Hp = hamming_matrix(P)Hd = hamming_matrix(D)print(f'distance matrices computed in {time.time()-t0:.1f}s')# upper triangle, cross-split onlyiu, ju = np.triu_indices(N, k=1)cross = split_arr[iu] != split_arr[ju]iu, ju = iu[cross], ju[cross]dp, dd = Hp[iu, ju], Hd[iu, ju]print(f'cross-split pairs examined: {len(iu):,}')MAX_KEEP = 8   # keep anything within the widest threshold we reportkeep = (dp <= MAX_KEEP) | (dd <= MAX_KEEP)pairs = pd.DataFrame(dict(    path_a=path_arr[iu[keep]], split_a=split_arr[iu[keep]], cls_a=cls_arr[iu[keep]],    path_b=path_arr[ju[keep]], split_b=split_arr[ju[keep]], cls_b=cls_arr[ju[keep]],    phash_dist=dp[keep], dhash_dist=dd[keep]))pairs['min_dist']   = pairs[['phash_dist','dhash_dist']].min(axis=1)pairs['same_class'] = pairs['cls_a'] == pairs['cls_b']pairs['split_pair'] = [' <-> '.join(sorted([a, b]))                       for a, b in zip(pairs['split_a'], pairs['split_b'])]pairs = pairs.sort_values('min_dist').reset_index(drop=True)pairs.to_csv(OUT/'dup_pairs.csv', index=False)print(f'wrote dup_pairs.csv: {len(pairs)} cross-split pairs with min_dist <= {MAX_KEEP}')

In [ ]:
def tally(thresh, col='min_dist'):    sub = pairs[pairs[col] <= thresh]    return subSUMMARY = dict(    n_images_hashed=int(len(df_ok)),    n_images_failed=int(len(failed)),    hash_size_bits=int(HASH_SIZE**2),    n_cross_split_pairs_examined=int(len(iu)),    per_split_counts={k: int(v) for k, v in df_ok.groupby('split').size().items()},    thresholds={},)print(f'{"threshold":>12} {"pairs":>8} {"exact":>8}  split-pair breakdown')print('-'*86)for th, name in [(0, 'exact (d=0)'), (5, 'd<=5'), (8, 'd<=8')]:    sub = tally(th)    by_pair = sub['split_pair'].value_counts().to_dict()    by_cls  = sub['cls_a'].value_counts().to_dict()    n_same  = int(sub['same_class'].sum())    # distinct test images implicated against train or val    test_hit = set()    for _, r in sub.iterrows():        if r['split_a'] == 'test' and r['split_b'] in ('train','val'): test_hit.add(r['path_a'])        if r['split_b'] == 'test' and r['split_a'] in ('train','val'): test_hit.add(r['path_b'])    SUMMARY['thresholds'][name] = dict(        n_pairs=int(len(sub)), n_same_class=n_same,        n_cross_class=int(len(sub)-n_same),        by_split_pair={k: int(v) for k, v in by_pair.items()},        by_class={k: int(v) for k, v in by_cls.items()},        n_test_images_with_dup_in_train_or_val=len(test_hit),        pct_of_test=round(100*len(test_hit)/max(1,int((df_ok['split']=='test').sum())), 3),    )    print(f'{name:>12} {len(sub):>8} {n_same:>8}  {by_pair}')print('\nPer-class breakdown (min_dist <= 5):')sub5 = tally(5)if len(sub5):    print(sub5.groupby(['split_pair','cls_a']).size().to_string())else:    print('  none')print('\nDistinct TEST images with a near-duplicate in train or val:')for name in SUMMARY['thresholds']:    t = SUMMARY['thresholds'][name]    print(f'  {name:>12}: {t["n_test_images_with_dup_in_train_or_val"]:4d} '          f'({t["pct_of_test"]:.2f}% of test)')with open(OUT/'dup_summary.json','w',encoding='utf-8') as f:    json.dump(SUMMARY, f, indent=2)print('\nwrote dup_summary.json')

## Montage of the 20 closest cross-split pairsFor manual inspection: a hash collision is not proof of duplication. Look at thesebefore deciding the leakage is real.

In [ ]:
top = pairs.head(20)if len(top) == 0:    print('No cross-split pairs within threshold -> no montage to draw.')else:    rows = len(top)    fig, axes = plt.subplots(rows, 2, figsize=(6, 3*rows))    if rows == 1:        axes = np.array([axes])    for r, (_, rec) in enumerate(top.iterrows()):        for c, side in enumerate(['a','b']):            ax = axes[r, c]            try:                with Image.open(rec[f'path_{side}']) as im:                    ax.imshow(im.convert('RGB'))            except Exception:                ax.text(0.5, 0.5, 'load failed', ha='center')            ax.set_title(f"{rec[f'split_{side}']} / {rec[f'cls_{side}']}\n"                         f"{Path(rec[f'path_{side}']).name[:28]}", fontsize=7)            ax.axis('off')        axes[r, 0].set_ylabel(f"p={rec['phash_dist']} d={rec['dhash_dist']}", fontsize=7)    plt.suptitle('20 closest cross-split pairs (left vs right) — inspect manually',                 fontsize=11, fontweight='bold')    plt.tight_layout()    plt.savefig(OUT/'montage_closest_pairs.png', dpi=130, bbox_inches='tight')    plt.close()    print(f'wrote montage_closest_pairs.png ({rows} pairs)')    print('\nClosest pairs:')    print(top[['split_a','cls_a','split_b','cls_b','phash_dist','dhash_dist']].to_string())

## Clean test manifestWrites `test_clean_manifest.csv` = the test images that have **no** near-duplicate(min_dist <= 5) in train or val. Notebook 1's final cell consumes this file andrecomputes every metric and CI on the clean subset.The threshold of 5 is the primary definition; `d<=8` is reported in`dup_summary.json` as a sensitivity check. If nothing is flagged, the file is stillwritten (containing the full test set) so Notebook 1 can confirm the null result.

In [ ]:
CLEAN_THRESHOLD = 5test_paths = df_ok.loc[df_ok['split']=='test','path'].tolist()contaminated = set()sub = pairs[pairs['min_dist'] <= CLEAN_THRESHOLD]for _, r in sub.iterrows():    if r['split_a'] == 'test' and r['split_b'] in ('train','val'):        contaminated.add(r['path_a'])    if r['split_b'] == 'test' and r['split_a'] in ('train','val'):        contaminated.add(r['path_b'])clean = [p for p in test_paths if p not in contaminated]cdf = pd.DataFrame(dict(path=clean))cdf['cls'] = [Path(p).parent.name for p in clean]cdf.to_csv(OUT/'test_clean_manifest.csv', index=False)print(f'test set              : {len(test_paths)}')print(f'flagged (d<={CLEAN_THRESHOLD} vs train/val): {len(contaminated)}')print(f'clean test manifest   : {len(clean)}')print('\nclean per class:')print(cdf['cls'].value_counts().to_string())print(f'\nwrote test_clean_manifest.csv')if len(contaminated) == 0:    print('\nNULL RESULT: no cross-split near-duplicates at this threshold.')    print('State this explicitly in the paper -- it is a positive finding.')else:    pct = 100*len(contaminated)/len(test_paths)    print(f'\nLEAKAGE FOUND: {pct:.2f}% of the test set has a near-duplicate in train/val.')    print('Re-run Notebook 1 section 15 and report BOTH numbers in the paper.')